# 00 — Harness check

Run this once after the one-time setup in the README. It proves three things before anyone relies on the harness:

1. **Data and features are identical to the team's.** The team's own `compare_intraday_anchors.py compare` stage already asserts the published original-anchor 15:30 results to 8 decimal places; this notebook re-reads that evidence.
2. **The harness reproduces the incumbent exactly.** Re-fitting the incumbent ridge through the harness must match, row by row, the 15:30 forecasts that the team's script computed from the same regenerated daily anchors.
3. **How close the regenerated incumbent is to the published one.** Re-fitting the tuned daily LightGBM is not bit-for-bit reproducible across machines, so the regenerated incumbent differs slightly from the published numbers. All sprint comparisons use the regenerated incumbent, so everyone shares one reference.

In [1]:
import sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, ROOT, ANCHOR_DIR, INCUMBENT, leaderboard
from notebooks.modeling_utils import forecast_metrics
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Data and features match the team's published run

In [2]:
summary = pd.read_json(ANCHOR_DIR / "verification_summary.json", typ="series")
print("Team script status:", summary["status"])
for check in summary["checks"]:
    print(" -", check)
assert summary["status"] == "passed"

Team script status: passed
 - common date/PERMNO keys and exact auction labels
 - unique keys and finite nonnegative daily forecasts
 - original annual daily and ridge scores reproduced
 - original checkpoint coverage reproduced
 - chronological update tuning and refitting
 - exact daily fallback for unavailable checkpoints
 - same historical samples across refitted ridge anchors
 - finite nonnegative ridge forecasts


## 2. The harness reproduces the incumbent row by row

In [3]:
h = Harness()
reference = pd.read_parquet(ANCHOR_DIR / "matched_predictions.parquet")[["date", "permno", "tree_daily", "tree_ridge_1530"]]
reference["date"] = pd.to_datetime(reference["date"])
rows = []
for fold in h.folds():
    correction, settings = h.incumbent_correction(fold)
    forecast = fold.to_shares(correction)
    mine = fold.test[["date", "permno"]].assign(harness=forecast, harness_daily=fold.test.anchor_forecast.to_numpy())
    joined = mine.merge(reference[reference.date.dt.year == fold.year], on=["date", "permno"], how="outer", indicator=True, validate="one_to_one")
    assert (joined["_merge"] == "both").all(), "Test rows differ from the team script"
    rows.append({
        "year": fold.year, "rows": len(joined), "ridge_strength": settings["strength"],
        "max_abs_diff_daily": np.abs(joined.harness_daily - joined.tree_daily).max(),
        "max_rel_diff_ridge": (np.abs(joined.harness - joined.tree_ridge_1530) / np.maximum(joined.tree_ridge_1530, 1)).max(),
        "harness_log_rmse": forecast_metrics(fold.test.auction_volume, forecast)["rmse_log"],
    })
check = pd.DataFrame(rows)
display(check)
assert (check.max_abs_diff_daily == 0).all()
assert (check.max_rel_diff_ridge < 1e-9).all(), "Harness does not reproduce the incumbent"
print("PASS: harness reproduces the team script's incumbent forecasts on every row.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,year,rows,ridge_strength,max_abs_diff_daily,max_rel_diff_ridge,harness_log_rmse
0,2022,327355,0.010000,0.000000,0.000000,0.495187
1,2023,316159,0.010000,0.000000,0.000000,0.445576
2,2024,312482,0.010000,0.000000,0.000000,0.422591
3,2025,304172,0.000100,0.000000,0.000000,0.406135


PASS: harness reproduces the team script's incumbent forecasts on every row.


## 3. Regenerated vs published incumbent

In [4]:
import io
# Published incumbent scores, read from the committed version of experiment 27 (the setup run overwrites the local copy).
text = subprocess.run(["git", "show", "HEAD:outputs/experiments/27_intraday_anchor_comparison/annual_metrics.csv"],
                      cwd=ROOT, capture_output=True, text=True, check=True).stdout
published = pd.read_csv(io.StringIO(text)).query("anchor == 'tree'")
published = published.pivot(index="year", columns="stage", values="rmse_log").add_prefix("published_")
regenerated = check.set_index("year")[["harness_log_rmse"]].rename(columns={"harness_log_rmse": "regenerated_ridge_1530"})
comparison = published.join(regenerated)
comparison["difference"] = comparison.regenerated_ridge_1530 - comparison.published_ridge_1530
display(comparison)
dev = comparison.loc[[2022, 2023, 2024]].mean()
print(f"Development log RMSE: published {dev.published_ridge_1530:.5f}, regenerated {dev.regenerated_ridge_1530:.5f}")

,published_daily,published_ridge_1530,regenerated_ridge_1530,difference
year,,,,
2022,0.504228,0.495390,0.495187,-0.000204
2023,0.454736,0.446750,0.445576,-0.001173
2024,0.433257,0.421976,0.422591,0.000615
2025,0.421605,0.404183,0.406135,0.001951


Development log RMSE: published 0.45471, regenerated 0.45445


## 4. Save the incumbent and smoke-test the scoring

This writes the incumbent's predictions once (so later reports are fast), then submits a deliberately weak model, a zero correction, which is just the daily anchor. The report must show it losing to the incumbent and failing the gate. That confirms the scoring path works end to end. The smoke test is not recorded on the leaderboard.

In [5]:
incumbent = h._incumbent_predictions()
print(f"Incumbent predictions: {len(incumbent):,} stock-days")
for fold in h.folds():
    h.submit(fold, np.zeros(len(fold.test)), name="smoke_zero_correction", owner="harness check",
             description="Zero correction = daily anchor; must lose to the incumbent")
result = h.report("smoke_zero_correction", record=False)
display(result["summary"])
display(result["intervals"])
display(pd.Series(result["gate"], name="gate"))
display(result["segments"])
assert not result["gate"]["passes_gate"]
print("PASS: scoring, intervals, gate and segments run end to end.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Incumbent predictions: 1,260,168 stock-days


smoke_zero_correction 2022: log RMSE 0.50648, WAPE 35.8932%, bias -16.31%


smoke_zero_correction 2023: log RMSE 0.46094, WAPE 34.5633%, bias -14.77%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (5 outputs left out)

,period,metric,difference,lower,upper
0,development,rmse_log,0.014590,0.013278,0.015943
1,development,wape,0.010351,0.009255,0.011491
2,development,signed_bias,-0.016814,-0.022169,-0.011800
3,development,absolute_bias,0.016814,0.011800,0.022169
4,confirmation,rmse_log,0.017024,0.014104,0.019488
5,confirmation,wape,0.007333,0.003827,0.010794
6,confirmation,signed_bias,-0.036209,-0.048134,-0.024645
7,confirmation,absolute_bias,0.036209,0.024645,0.048134


dev_rmse_gain       -0.014590
gain_ok                 False
worst_year_change    0.017112
annual_ok               False
wape_ok                 False
interval_ok             False
passes_gate             False
Name: gate, dtype: object

model,daily_anchor,incumbent_tree_ridge,smoke_zero_correction
segment,,,
all,0.465554,0.456067,0.470425
fomc_day,0.495698,0.479601,0.497969
largest_1pct_auctions (diagnostic),1.110459,1.068126,1.110824
low_liquidity,0.521303,0.523448,0.535439
month_end,0.656296,0.652592,0.658039
msci_review,0.778635,0.773435,0.780287
quarterly_expiry,0.761039,0.730799,0.761123
russell_recon,1.027303,1.011185,1.028790
short_history,1.017115,0.997037,1.016907


PASS: scoring, intervals, gate and segments run end to end.
